## Imports and Inititalizations

In [4]:
import pandas as pd
import base64
import os
import requests
from io import BytesIO
from PIL import Image
import nltk
import json
import itertools
import time

# 1. API Key Rotation Setup (Using your .env or a list)
# Assuming you have an .env file with one key per line, similar to the Gemma notebook
env_path = ".env" 
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
else:
    print(f"Error: '{env_path}' file not found. Falling back to manual list.")
    API_KEYS = ["YOUR_OPENROUTER_KEY_1", "YOUR_OPENROUTER_KEY_2"]

key_cycle = itertools.cycle(API_KEYS)
print(f"Successfully loaded {len(API_KEYS)} API keys for rotation.")

# 2. Evaluator Setup
from doc_parsing_evaluator import ParsingEvaluator
nltk.download('punkt')

# 3. Path Configuration (Based on your folder screenshot)
BASE_DIR = r"C:\Users\anshg\OneDrive\Desktop\BTP\Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing\CC-OCR_Dataset\doc_parsing"
OUT_DIR = r"C:\Users\anshg\OneDrive\Desktop\BTP\Evaluation-Of-MultiModal-LLMs-for-Layout-Aware-Document-Parsing\Evaluation_Results"
os.makedirs(OUT_DIR, exist_ok=True)

# 4. Model Selection
MODEL_NAME = "qwen/qwen2.5-vl-72b-instruct"

evaluator = ParsingEvaluator(group_name="OCR_Eval")

Successfully loaded 9 API keys for rotation.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anshg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Inference Engine (Qwen2.5vl:72b)

In [2]:
def query_qwen(image_bytes, task_type):
    """Sends image to Qwen via OpenRouter, rotating through API keys."""
    url = "https://openrouter.ai/api/v1/chat/completions"
    current_key = next(key_cycle)
    
    if task_type == "table":
        prompt = "Parse this table into a clean, structured HTML format with <table> tags. Include all row and column spans."
    else:
        prompt = "Parse this document into LaTeX format. Provide only the LaTeX code."

    image_base64 = base64.b64encode(image_bytes).decode("utf-8")

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{image_base64}"}
                    }
                ]
            }
        ]
    }

    headers = {
        "Authorization": f"Bearer {current_key}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost", # Required by OpenRouter
        "X-Title": "BTP-OCR-Evaluation"
    }

    try:
        response = requests.post(url, headers=headers, json=payload, timeout=60)
        result = response.json()
        
        if "choices" in result:
            return result["choices"][0]["message"]["content"]
        else:
            print(f"Error with key {current_key[:8]}...: {result.get('error', 'Unknown error')}")
            return ""
    except Exception as e:
        print(f"Request failed: {e}")
        return ""

## Document Parsing Evaluation (LaTeX)

In [5]:
doc_tasks = {
    "Scanned": os.path.join(BASE_DIR, "doc", "doc_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "doc", "doc_photo_eng_75.tsv")
}

doc_results = {}

for label, path in doc_tasks.items():
    if not os.path.exists(path): continue
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} documents...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth = str(row['answer']).strip() 
        
        prediction = query_qwen(img_bytes, "doc")
        
        score = evaluator.evaluate_single_doc_sample(ground_truth, prediction)
        scores.append(score)
        
        # Rate limiting buffer for free tier
        time.sleep(2) 
        
    doc_results[label] = sum(scores) / len(scores) if scores else 0

print(f"Document Parsing (NED with LaTeX) Results: {doc_results}")

Processing Scanned documents...
Error with key sk-or-v1...: {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}
Error with key sk-or-v1...: {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrouter.ai/settings/credits', 'code': 402}
Error with key sk-or-v1...: {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 65536 tokens, but can only afford 15606. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'provider_name': None}}
Error with key sk-or-v1...: {'message': 'Insufficient credits. This account never purchased credits. Make sure your key is on the correct account or org, and if so, purchase more at https://openrou

KeyboardInterrupt: 

## Table Parsing Evaluation (HTML)

In [ ]:
table_tasks = {
    "Scanned": os.path.join(BASE_DIR, "table", "table_scan_eng_75.tsv"),
    "Photoed": os.path.join(BASE_DIR, "table", "table_photo_eng_75.tsv")
}

table_results = {}

for label, path in table_tasks.items():
    if not os.path.exists(path): continue
        
    df = pd.read_csv(path, sep='\t')
    scores = []
    
    print(f"Processing {label} tables...")
    
    for _, row in df.iterrows():
        img_bytes = base64.b64decode(row['image'])
        ground_truth_html = str(row['answer']).strip() 
        
        prediction = query_qwen(img_bytes, "table")
        
        score = evaluator.evaluate_single_table_sample(ground_truth_html, prediction)
        scores.append(score)
        
        time.sleep(2)

    table_results[label] = sum(scores) / len(scores) if scores else 0

print(f"\nTable Parsing (TEDS Results): {table_results}")

Processing Scanned tables...
Processing Photoed tables...

Table Parsing (TEDS Results): {'Scanned': 0.6555066174747228, 'Photoed': 0.5207718393466946}


## Final Report 

In [ ]:
summary = {
    "Document Parsing (NED-LaTeX)": doc_results,
    "Table Parsing (HTML-Sim)": table_results,
    "Model": MODEL_NAME,
    "Track": "English Document Parsing"
}

with open(os.path.join(OUT_DIR, "stage1_final_report.json"), "w") as f:
    json.dump(summary, f, indent=4)

print("Evaluation Complete. Check 'stage1_final_report.json' for full details.")

Evaluation Complete. Check 'stage1_final_report.json' for full details.
